# Conserta Aí — Avaliação da IA
Notebook reprodutível da avaliação do serviço **ia-visao**:
1. **Visão computacional** – CLIP zero-shot x linear probe (acurácia, F1 macro, matriz de confusão)
2. **IA generativa** – triagem com LLM (acerto de categoria, prioridade ±1, JSON válido, casos de falha)

**Como rodar no Google Colab:** `Ambiente de execução > Executar tudo`.
Antes, salve sua chave do Gemini em 🔑 *Secrets* (barra lateral) com o nome `GEMINI_API_KEY`.
Chave grátis: https://aistudio.google.com/apikey

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/poiuytrewq01/conserta-ai-ia.git"  # troque se o repositório tiver outro endereço

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB and not os.path.exists("avaliacao"):
    if not os.path.exists("conserta-ai-ia"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("conserta-ai-ia")
if os.path.basename(os.getcwd()) == "avaliacao":
    os.chdir("..")
print("Pasta do projeto:", os.getcwd())

In [ ]:
# Colab já vem com torch; instala o resto
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-avaliacao.txt"], check=True)

In [ ]:
# Chave do LLM: Secrets do Colab, .env local ou digitada na hora (nunca fica salva no notebook)
if not os.getenv("LLM_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["LLM_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        from dotenv import load_dotenv; load_dotenv()
if not os.getenv("LLM_API_KEY"):
    from getpass import getpass
    os.environ["LLM_API_KEY"] = getpass("Cole a chave do Gemini: ")
print("Chave configurada:", bool(os.getenv("LLM_API_KEY")))

## 1. Dados de teste (fotos com fonte e licença)

In [ ]:
subprocess.run([sys.executable, "avaliacao/baixar_fotos.py", "--por-categoria", "20"], check=True)

In [ ]:
# Galeria para conferir as fotos. Se alguma estiver errada, coloque o caminho dela
# em avaliacao/dados/fotos_excluidas.txt (uma por linha) e rode a avaliação de novo.
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
man = pd.read_csv("avaliacao/dados/manifest_fotos.csv")
display(man.groupby(["conjunto", "categoria"]).size().rename("fotos").to_frame())
for pasta, grupo in man.groupby("pasta"):
    amostra = grupo.head(8)
    fig, axs = plt.subplots(1, len(amostra), figsize=(2.2 * len(amostra), 2.4))
    for ax, (_, l) in zip(list(axs) if len(amostra) > 1 else [axs], amostra.iterrows()):
        ax.imshow(Image.open("avaliacao/dados/" + l["arquivo"]).convert("RGB")); ax.axis("off")
    fig.suptitle(pasta); plt.show()

## 2. Visão computacional — CLIP zero-shot x linear probe

In [ ]:
subprocess.run([sys.executable, "avaliacao/avaliar_visao.py"], check=True)
from IPython.display import Markdown, Image as Img, display
display(Markdown(open("avaliacao/resultados/visao.md", encoding="utf-8").read().split("![")[0]))
display(Img("avaliacao/resultados/visao_matriz_zeroshot.png"))
if os.path.exists("avaliacao/resultados/visao_matriz_linear_probe.png"):
    display(Img("avaliacao/resultados/visao_matriz_linear_probe.png"))

## 3. IA generativa — triagem com LLM

In [ ]:
subprocess.run([sys.executable, "avaliacao/avaliar_llm.py", "--pausa", "4"], check=True)
display(Markdown(open("avaliacao/resultados/llm.md", encoding="utf-8").read()))

In [ ]:
res = pd.read_csv("avaliacao/resultados/llm_resultados.csv")
res[["id", "tipo", "descricao", "categoria_prevista", "prioridade_prevista", "resumo_tecnico", "origem"]]

## 4. Demonstração do fluxo completo (foto + descrição → visão → LLM)

In [ ]:
sys.path.insert(0, "ia-visao")
from app.visao import ClassificadorClip
from app.triagem import Triador
clf, tri = ClassificadorClip(), Triador()
foto = "avaliacao/dados/" + man[man.pasta == "buraco"].iloc[0]["arquivo"]
v = clf.classificar(Image.open(foto).convert("RGB"))
t = tri.triar("Buraco grande na frente da escola, carro quase caiu", "Buraco",
              v["categoria"] if v["confiavel"] else None, v["confianca"] if v["confiavel"] else None)
display(Img(foto, width=300)); print("VISÃO:", v); print("TRIAGEM:", t)

## 5. Baixar os resultados
Suba a pasta `avaliacao/resultados/` para o GitHub (é a evidência da avaliação).

In [ ]:
import shutil
shutil.make_archive("resultados", "zip", "avaliacao/resultados")
if EM_COLAB:
    from google.colab import files; files.download("resultados.zip")